# 01 — Mean-variance: minimum variance (GMV) and maximum Sharpe ratio (MSR)

GMV and MSR as Python classes (`GMV` and `MaxSharpe` in `maplab.models`; the text writes MSR, code and figures `MaxSharpe`), scored by the rolling backtest against the two benchmarks used in every notebook: equal weight (EW) and 60/40 (SPY/IEF).

## §1 Why mean-variance is fragile

GMV is risk-based: it needs only the covariance Σ. MSR is return-based: it also needs a forecast of expected returns μ, here the trailing one-year mean. Because μ and Σ are estimates, mean-variance overweights the assets whose μ is overestimated, and a small change in μ moves the weights a lot. That is why it is called an error maximizer. Errors in μ cost far more than errors in Σ, which is why simple benchmarks are hard to beat, and why every model here faces EW and 60/40. Notebook 03 tries a better μ, 04 a better Σ, and 05–07 drop μ altogether.

## §2 The optimization problems

Unconstrained, both portfolios have closed forms:

$$w_{\text{GMV}} = \frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}'\Sigma^{-1}\mathbf{1}}, \qquad w_{\text{MSR}} = \frac{\Sigma^{-1}(\mu - r_f\mathbf{1})}{\mathbf{1}'\Sigma^{-1}(\mu - r_f\mathbf{1})}.$$

On this universe the unconstrained MSR takes large offsetting long and short positions (§4.2), so the classes solve the long-only versions:

$$\text{GMV:}\quad \min_{w}\; w'\Sigma w \quad \text{s.t.}\quad \mathbf{1}'w = 1,\; w \ge 0,$$

$$\text{MSR:}\quad \max_{w}\; \frac{(\mu - r_f\mathbf{1})'w}{\sqrt{w'\Sigma w}} \quad \text{s.t.}\quad \mathbf{1}'w = 1,\; w \ge 0.$$

**Inputs**, from the 252 trading days before each rebalance: $\Sigma$ is the annualized sample covariance of log returns; $\mu = \overline{r}_{\log} \times 252 + \tfrac12\,\mathrm{diag}(\Sigma)$ converts the log mean into an arithmetic mean; $r_f$ is the trailing mean of BIL.

## Setup

The 13 ETFs, BIL as the risk-free rate and the conventions of notebook 00; log returns feed μ and Σ, simple returns compound the backtest.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import maplab as ml
import _nb01_helpers as h
from maplab import Panel, apply_style
from maplab.models import tangency_closed_form

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf = ml.load_rf_returns()                      # BIL, risk-free only
rf_daily = rf["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": rf})

## §3 The classes and the rolling backtest

Calling a model at a rebalance date runs three steps: estimate μ and Σ from the 252 trading days before the date (the §2 inputs; `Strategy._estimate` in `src/maplab/models.py`), solve the §2 problem, and validate the weights (no missing values, sum to 1, no negative weights for long-only classes). Every model in this repo shares the first and last steps.

**Solvers.** Both long-only problems are solved numerically. MSR uses the convex form $\min_y y'\Sigma y$ s.t. $(\mu - r_f\mathbf{1})'y = 1$, $y \ge 0$, then sets $w = y/\mathbf{1}'y$. If no asset has $\mu_i > r_f$, there is no positive Sharpe ratio to maximize and the class falls back to GMV.

**Backtest** (`ml.backtest`), identical for every model: rebalance at each month-end using only earlier data (2022-12-31 uses data through 12-30 and trades on 2023-01-03); pay 10 bp per unit of one-way turnover; let weights drift between rebalances; score from 2009-01-31, 208 rebalances in all.

## §4 Snapshot at the train/test split

One rebalance date, 2022-12-31: what the classes see and what they do with it.

### §4.1 Inputs at the snapshot

The estimation window, $r_f$, μ and volatility per asset, and UUP's correlations on the 252 trading days before the split.

In [ ]:
from maplab import GMV, MaxSharpe

rdates = ml.rebalance_dates(simple_returns.index)
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)
split_rebal = rdates[rdates <= split_ts].max()

mu, Sigma = GMV()._estimate(panel, split_rebal)
rf_ann = MaxSharpe()._rf_ann(panel, split_rebal)
window = panel.slice(split_rebal, "returns", 252)

print(f"snapshot date: {split_rebal.date()}")
print(f"estimation window: {window.index.min().date()} -> {window.index.max().date()} "
      f"({len(window)} trading days)")
print(f"rf (trailing 252-day BIL mean, annualized): {rf_ann:.2%}\n")

vol = pd.Series(np.sqrt(np.diag(Sigma.to_numpy())), index=Sigma.index)
inputs = pd.DataFrame({"μ (%)": mu * 100, "vol (%)": vol * 100, "μ − r_f (%)": (mu - rf_ann) * 100}).rename_axis(None)
print(inputs.sort_values("μ − r_f (%)", ascending=False).to_string(float_format=lambda v: f"{v:.2f}"))

corr = Sigma / np.outer(vol, vol)
print("\nUUP's correlation with the other 12 (252-day window):")
print(corr["UUP"].drop("UUP").sort_values().rename_axis(None).to_string(float_format=lambda v: f"{v:.2f}"))
print(f"\ncorr(EFA, EEM): {corr.loc['EFA', 'EEM']:.2f}")

**Reading.** The window is the 2022 rate shock. Only DBC (+19.3%) and UUP (+7.6%) have μ above $r_f$; GLD sits just below it (−0.3%) and the other ten are negative. UUP also has the lowest volatility (9.27%; TIP 9.34%) and is negatively correlated with every other asset (−0.22 to −0.75). MSR allocates as if 2022's average returns continue.

### §4.2 Long-only vs unconstrained weights

The classes' long-only weights next to the closed forms of §2 on the same inputs. The tangency closed form exists only if the unconstrained GMV's expected return exceeds $r_f$, which is checked first.

In [ ]:
from maplab.models import gmv_closed_form

w_gmv = GMV()(panel, split_rebal)
w_msr = MaxSharpe()(panel, split_rebal)
w_gmv_unc = gmv_closed_form(Sigma)

mu_gmv_unc = float(w_gmv_unc @ mu)
print(f"unconstrained GMV expected return: {mu_gmv_unc:.2%} vs rf {rf_ann:.2%} "
      f"-> closed-form tangency defined: {mu_gmv_unc > rf_ann}\n")
w_tan = tangency_closed_form(mu, Sigma, rf=rf_ann)

weights = pd.DataFrame({
    "GMV long-only": w_gmv,
    "MSR long-only": w_msr,
    "GMV unconstrained": w_gmv_unc,
    "MSR unconstrained": w_tan,
}).rename_axis(None)
print("weights (%):")
print((weights * 100).sort_values("MSR unconstrained", ascending=False)
      .to_string(float_format=lambda v: f"{v:.1f}"))

The four portfolios as bars (grey: unconstrained closed form; colour: the class's long-only weights; own y-axis per panel), with gross leverage, total short weight and largest absolute weight below.

In [ ]:
fig, leverage_table = h.unconstrained_vs_long_only(Sigma, mu, rf_ann, w_gmv, w_msr)
plt.show()

lt = leverage_table
shown = lt.astype(object)
shown.loc["gross leverage Σ|w|"] = lt.loc["gross leverage Σ|w|"].map(lambda v: f"{v:.2f}×")
for row in ("sum of negative weights", "max |w|"):
    shown.loc[row] = (lt.loc[row] * 100).map(lambda v: f"{v:.1f}%")
print(shown.to_string())

**Reading.** Long-only GMV holds UUP 58%, EFA 17%, IEF 14% and GLD 11%; long-only MSR holds UUP 74%, DBC 23% and GLD 3%. Unconstrained, MSR runs 13.0× gross leverage: +193% UUP and +165% EFA against −216% TIP and −126% EEM. $\Sigma^{-1}$ turns small gaps in μ between highly correlated assets into large offsetting bets: EFA and EEM (correlation 0.82) differ by 7.6 points of μ and end up at +165% and −126%, while unconstrained GMV, with no μ to bet on, stays at 1.6×. That is estimation error at full strength, and the reason the classes are long-only.

### §4.3 Sensitivity to μ

Perturb μ with draws from $N(0, s^2\Sigma)$ and re-solve MSR, 200 draws per noise level $s$. That is the sampling error of a one-year mean: the standard errors equal the annualized volatilities, and the errors are correlated like the returns. GMV does not use μ and is unchanged by construction.

In [ ]:
from maplab.models import _tangency_long_only

n_draws = 200
w_msr_base = w_msr.to_numpy()
Sigma_np = Sigma.to_numpy()

rows = []
for scale in (0.25, 0.5, 1.0):
    rng = np.random.default_rng(20260924)
    for _ in range(n_draws):
        eps = rng.multivariate_normal(np.zeros(len(mu)), Sigma_np)  # one-year sampling error of mu: N(0, Sigma)
        excess = mu.to_numpy() + scale * eps - rf_ann
        if excess.max() <= 0:
            continue
        w_m, _ = _tangency_long_only(excess, Sigma_np, "MSR", split_rebal)
        rows.append({
            "noise": scale,
            "half_L1": 0.5 * np.abs(w_m - w_msr_base).sum(),
            "max_w": w_m.max(),
            "holdings": int((w_m > 1e-4).sum()),
            "top": mu.index[int(np.argmax(w_m))],
        })
sens = pd.DataFrame(rows)

table = sens.groupby("noise").agg(
    draws=("half_L1", "size"),
    half_L1_median=("half_L1", "median"),
    half_L1_p90=("half_L1", lambda s: s.quantile(0.9)),
    max_w_median=("max_w", "median"),
    holdings_median=("holdings", "median"),
    UUP_top_share=("top", lambda s: (s == "UUP").mean()),
)
print("MSR weight change vs the unperturbed solution (noise in units of the one-year std error of mu):")
shown = pd.DataFrame({
    "draws": table["draws"].map(lambda v: f"{v:d}"),
    "median ½L1 (%)": (table["half_L1_median"] * 100).map(lambda v: f"{v:.1f}"),
    "p90 ½L1 (%)": (table["half_L1_p90"] * 100).map(lambda v: f"{v:.1f}"),
    "median max weight (%)": (table["max_w_median"] * 100).map(lambda v: f"{v:.1f}"),
    "median holdings": table["holdings_median"].map(lambda v: f"{v:.0f}"),
    "UUP on top (%)": (table["UUP_top_share"] * 100).map(lambda v: f"{v:.1f}"),
})
shown.index = shown.index.map(lambda s: f"{s:.2f}")
shown.index.name = "s"
print(shown.to_string())

**Reading.** At one standard error ($s = 1$) the median draw reallocates 25% of MSR's portfolio (53% at the 90th percentile), and UUP loses the top spot in 15% of draws; 199 of the 200 draws count, because one left no asset above $r_f$. At a quarter of that noise the median draw still reallocates 8%, and UUP stays on top in every draw. The median draw holds three assets at every level. This is the error maximizer of §1, and with μ re-estimated every month it shows up as turnover in §5.

### §4.4 The long-only efficient frontier

The long-only frontier on the §4.1 inputs, the capital market line through MSR, and the benchmarks. The grey cloud is 20,000 random long-only portfolios drawn from a Dirichlet(0.3); α < 1 concentrates the draws, so the cloud reaches the corners where GMV and MSR sit.

In [ ]:
mc_alpha = 0.3
fig = h.frontier_figure(Sigma, mu, w_gmv, w_msr, mc_alpha, rf_ann, split_rebal)
plt.show()

**Reading.** On 2022's inputs EW and 60/40 sit inside the cloud with negative expected returns, while MSR sits where the capital market line touches the frontier. That ranking is only as good as one year of μ; §5 shows what happened next.

### §4.5 Capital vs risk

Capital weights (dark) against each asset's share of portfolio variance (light) at 2022-12-31; a negative share means the asset hedges the rest. The table counts diversification three ways: effective N $= 1/\sum_i w_i^2$, the number of equally weighted assets with the same concentration; uncorrelated bets, the exponential of the entropy of the portfolio's variance shares across Σ's principal portfolios (between 1 and 13); and the diversification ratio $w'\sigma/\sigma_p$. For long-only GMV the two bars coincide for every held asset: at the optimum every held asset has the same marginal risk, so its variance share equals its weight.

In [ ]:
from maplab import EqualWeight, FixedWeight

weights_snap = {
    "GMV": w_gmv,
    "MaxSharpe": w_msr,
    "EW": EqualWeight()(panel, split_rebal),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40")(panel, split_rebal),
}
div = ml.diagnostics.diversification_table(weights_snap, Sigma)
div_shown = pd.DataFrame({
    "UUP weight (%)": div["w_UUP"] * 100,
    "effective N": div["effN"],
    "uncorrelated bets": div["n_uncorr"],
    "largest risk share (%)": div["max_RC_share"] * 100,
    "diversification ratio": div["DR"],
    "ex-ante vol (%)": div["exante_vol"] * 100,
})
print(div_shown.to_string(formatters={
    "UUP weight (%)": "{:.1f}".format, "effective N": "{:.2f}".format, "uncorrelated bets": "{:.2f}".format,
    "largest risk share (%)": "{:.1f}".format, "diversification ratio": "{:.2f}".format, "ex-ante vol (%)": "{:.1f}".format}))

fig = ml.plotting.capital_vs_risk(weights_snap, Sigma, title=f"Capital weight vs risk share, {split_rebal.date()} (sample Σ)")
plt.show()

**Reading.** Holding many assets is not holding many bets: EW's 13 assets amount to about 1.4 uncorrelated bets, and SPY carries 89% of 60/40's variance for 60% of its capital. MSR concentrates both ways (UUP is 74% of capital and 57% of variance), while GMV has the highest diversification ratio (3.69) and about 7.6 uncorrelated bets from four assets. The bets are counted over principal portfolios, not assets, so they can exceed the number of holdings.

## §5 Out-of-sample results

Walk-forward, as set out in notebook 00: at each month-end the class re-estimates μ and Σ on the prior 252 days and holds the weights for a month, so every return is out of sample; "train" and "test" split one track record at 2022-12-31. The cell checks the Sharpe ratios against `registrations/reproduction.toml`, the values later notebooks build on.

In [ ]:
from maplab import EqualWeight, FixedWeight, backtest, summary

strategies = {
    "GMV": GMV(),
    "MaxSharpe": MaxSharpe(),
    "EqualWeight": EqualWeight(),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40"),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag}

rows = []
for name, r in results.items():
    net = r["net"]
    turnover = r["diag"]["turnover_per_rebalance"]
    windows = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in windows.items():
        s = summary(net.loc[net_sel], turnover.loc[t_sel], rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]

msr = strategies["MaxSharpe"]
full = summary_table.xs("full", level="window")
train = summary_table.xs("train (<= split)", level="window")
test = summary_table.xs("test (> split)", level="window")
overview = pd.DataFrame({
    "Sharpe full": full["sharpe"], "Sharpe train": train["sharpe"], "Sharpe test": test["sharpe"],
    "return (%)": full["ann_return"] * 100, "vol (%)": full["ann_vol"] * 100,
    "max DD (%)": full["max_dd"] * 100, "turnover (%/yr)": full["ann_turnover"] * 100,
}).rename(index={"EqualWeight": "EW"}).rename_axis(None)
print(overview.to_string(formatters={
    "Sharpe full": "{:.2f}".format, "Sharpe train": "{:.2f}".format, "Sharpe test": "{:.2f}".format,
    "return (%)": "{:.1f}".format, "vol (%)": "{:.1f}".format, "max DD (%)": "{:.1f}".format,
    "turnover (%/yr)": "{:.0f}".format}))

ml.inference.check_reproduction(summary_table, {
    "GMV": "GMV_sample", "MaxSharpe": "MaxSharpe_sample",
    "EqualWeight": "EW", "60/40": "SixtyForty_SPY_IEF",
})
print("\nreproduction gate passed (GMV, MaxSharpe, EW, 60/40 vs registrations/reproduction.toml, 2 dp); "
      f"MaxSharpe fallbacks {len(msr.fallback_dates)}, retries {len(msr.retry_dates)}")

**Reading.** Over the full sample 60/40 has the highest Sharpe ratio (0.91), ahead of EW (0.74), MSR (0.72) and GMV (0.62). From 2023 MSR leads (1.13; 60/40 1.00, EW 0.91, GMV 0.76), but that window is 40 months. GMV has the lowest volatility and 60/40 the highest (3.2% vs 10.1%). MSR turns over 248% a year, which at 10 bp costs about 0.25% a year against a 1.2-point return gap to EW (5.8% vs 7.0%), so costs explain little of its lag. Whether any of these gaps is significant is tested in notebook 08.

## §6 What drove the results: allocation through time

### §6.1 Weights by asset group

Target weights summed into the six groups at every rebalance (dotted line: train/test split). EW and 60/40 are fixed and appear only in the table.

In [ ]:
gmv_group = ml.diagnostics.weights_by_group(results["GMV"]["wlog"])
msr_group = ml.diagnostics.weights_by_group(results["MaxSharpe"]["wlog"])

fig = ml.plotting.group_stackplot(results, ["GMV", "MaxSharpe"], figsize=(9, 8),
                                  title_fmt="{name} — target weights by asset group", split_ts=split_ts)
plt.show()

print("Mean target weight by asset group (%):")
group_means = pd.DataFrame({
    "GMV": gmv_group.mean(),
    "MaxSharpe": msr_group.mean(),
    "EW": ml.diagnostics.weights_by_group(results["EqualWeight"]["wlog"]).mean(),
    "60/40": ml.diagnostics.weights_by_group(results["60/40"]["wlog"]).mean(),
}).rename_axis(None)
print((group_means * 100).to_string(float_format=lambda v: f"{v:.1f}"))

uup_w = results["GMV"]["wlog"]["UUP"]
print(f"\nGMV UUP weight: mean {uup_w.mean():.1%}, min {uup_w.min():.1%}, max {uup_w.max():.1%}")

msr_peaks = pd.DataFrame({"max weight (%)": msr_group.max() * 100,
                          "first date at max": msr_group.idxmax().dt.date}).rename_axis(None)
print("\nMaxSharpe: peak weight by group")
print(msr_peaks.to_string(float_format=lambda v: f"{v:.1f}"))
near_all = msr_group[msr_group.max(axis=1) >= 0.95]
print(f"\nMaxSharpe rebalances with one group >= 95%: {len(near_all)} of {len(msr_group)}")
for d, row in near_all.iterrows():
    print(f"  {d.date()}  {row.idxmax()}  {row.max():.1%}")

**Reading.** GMV is a dollar-and-bonds portfolio: 37% UUP on average (18–58%), 21% rates, 16% TIPS, 13% credit and only 7% equities, against 23% for EW and 60% for 60/40. Low volatility is its objective; the low return goes with holding so little equity. MSR's averages look diversified but hide the path: it was 95–100% in equities from February to April 2014 and 100% in August 2017, and peaked at 85% in the dollar in October 2022. Because μ is each asset's trailing one-year mean (§2), MSR rotates into whatever had the best risk-adjusted year, with nothing to smooth the switch.

### §6.2 Concentration and turnover

Effective N, largest weight and trailing-12-rebalance turnover at every rebalance; EW dashed in the effective-N and turnover panels.

In [ ]:
names = ["GMV", "MaxSharpe", "60/40"]
colors = {name: ml.FAMILY_COLORS[strategies[name].family] for name in ["GMV", "MaxSharpe"]}
colors["60/40"] = ml.plotting.METHOD_COLORS["60/40"]

fig = ml.plotting.concentration_panel(
    results, names,
    rows=("effN", "max_w", "turnover"), ref="EqualWeight", split_ts=split_ts,
    figsize=(9.5, 10.5), colors=colors,
    suptitle="GMV, MaxSharpe and 60/40: effective N, largest weight, turnover",
)
plt.show()

conc = pd.DataFrame({
    name: {
        "median effN": results[name]["wlog"].apply(ml.diagnostics.effective_n, axis=1).median(),
        "median max weight (%)": results[name]["wlog"].max(axis=1).median() * 100,
        "largest max weight (%)": results[name]["wlog"].max(axis=1).max() * 100,
        "ann. turnover (%/yr)": summary_table.loc[(name, "full"), "ann_turnover"] * 100,
    }
    for name in ["GMV", "MaxSharpe", "EqualWeight", "60/40"]
}).T.rename(index={"EqualWeight": "EW"})
print(conc.to_string(formatters={
    "median effN": "{:.2f}".format, "median max weight (%)": "{:.1f}".format,
    "largest max weight (%)": "{:.1f}".format, "ann. turnover (%/yr)": "{:.0f}".format}))

**Reading.** Both optimizers are concentrated (median effective N 3.8 for GMV, 3.0 for MSR, against 13 for EW), but 60/40 is more concentrated than either (1.9) and has the best Sharpe (§5): a low effective N does not by itself mean a worse result. What separates GMV from MSR is stability. GMV's largest weight never exceeds 58%, while MSR's reaches 100%, and MSR pays for that with 248% annual turnover. Both converge on UUP into the 2022 split: GMV's UUP weight reaches its 58% maximum there, and MSR's fx weight its 85% peak (§6.1).

## §7 TAKEAWAY — the benchmarks win on Sharpe; the optimizers differ in risk and stability

- **The fixed-weight benchmarks are hard to beat.** Full-sample Sharpe: 60/40 0.91, EW 0.74, MSR 0.72, GMV 0.62; MSR leads only in the 40-month test window (1.13).
- **Risk ranks clearly; cost barely matters.** GMV has the lowest volatility and 60/40 the highest (3.2% vs 10.1%); MSR's 248% annual turnover costs about 0.25% a year.
- **MSR is an estimation-error maximizer.** One standard error of noise in μ moves a median 25% of its weights (§4.3); unconstrained it runs 13× gross, so long-only is not cosmetic.
- **GMV leans on the dollar.** UUP averages 37% of GMV (18–58%), with 7% in equities; it enters as a hedge, negatively correlated with all 12 other assets at the snapshot (§4.1).
- **Many assets are not many bets.** EW's 13 assets are about 1.4 uncorrelated bets, yet 60/40, the most concentrated portfolio, has the best Sharpe; MSR's weakness is instability, not concentration (§6.2).

Sample-mean μ is the weak link: notebook 03 tries a better μ, notebook 04 a better Σ, and notebooks 05–07 drop μ altogether.